# 🚗 UA-DETRAC Trafik Veri Seti ile YOLO11 Araç Tespiti Eğitimi
### Bitirme Projesi - 45.000+ Gerçek Araç Etiketi ile Yüksek Doğruluklu Model Eğitimi

Bu notebook, **UA-DETRAC Benchmark Trafik Veri Seti (5.640 görsel, 45.000+ araç)** kullanılarak **YOLO11** modelini GPU hızlandırıcısı ile eğitmek için hazırlanmıştır.

**Kritik Adım:** Hızlı eğitim için Colab'in **T4 GPU** modunda olması gerekir:
👉 **Çalışma zamanı (Runtime) -> Çalışma zamanı türünü değiştir (Change runtime type) -> T4 GPU** seçip Kaydet'e basın.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aydin-eng/federated-traffic-learning/blob/main/traffic_vehicle_detection_yolo11.ipynb)


## 1. Donanım ve GPU Kontrolü
GPU'nun (T4) devrede olduğunu doğruluyoruz.


In [14]:
import torch

if torch.cuda.is_available():
    print(f'✅ Harika! GPU devrede: {torch.cuda.get_device_name(0)}')
    !nvidia-smi
else:
    print('⚠️ UYARI: GPU devrede DEĞİL, CPU modundasınız!')
    print('Lütfen menüden: Çalışma zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU seçip Kaydet deyin.')


✅ Harika! GPU devrede: Tesla T4
Wed Nov 11 17:10:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------

## 2. Gerekli Kütüphanelerin Yüklenmesi
Ultralytics (YOLO11) kütüphanesini kuruyoruz.


In [15]:
!pip install -q ultralytics


## 3. GitHub Deposunu Klonlama
Test fotoğrafları ve yardımcı araçları çekiyoruz.


In [3]:
import os

if not os.path.exists('federated-traffic-learning'):
    !git clone https://github.com/aydin-eng/federated-traffic-learning.git
    %cd federated-traffic-learning
else:
    %cd federated-traffic-learning
    !git pull

print('Mevcut çalışma dizini:', os.getcwd())


Cloning into 'federated-traffic-learning'...
remote: Enumerating objects: 819, done.
remote: Counting objects: 100% (302/302), done.
remote: Compressing objects: 100% (275/275), done.
remote: Total 819 (delta 26), reused 285 (delta 14), pack-reused 517 (from 1)
Receiving objects: 100% (819/819), 130.91 MiB | 23.98 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/federated-traffic-learning
Mevcut çalışma dizini: /content/federated-traffic-learning


## 4. UA-DETRAC Veri Setinin Yüklenmesi (`ua_detrac_yolo.zip`)
Hazırladığımız **`ua_detrac_yolo.zip` (374 MB)** dosyasını Colab'e bağlıyoruz.

**2 Kolay Yoldan Birini Seçin:**
* **Seçenek A (Google Drive - Önerilen):** 4.1 hücresini çalıştırın.
* **Seçenek B (Doğrudan Yükleme):** 4.2 hücresini çalıştırıp dosyayı seçin.


### 4.1 Seçenek A: Google Drive Bağlama (Önerilen)
`ua_detrac_yolo.zip` dosyasını Google Drive'ınızın ana sayfasına sürükleyip attıysanız bu hücreyi çalıştırın.


In [12]:
import os
import glob
from google.colab import drive

# Google Drive'ı bağla
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Drive içinde doğru dosyayı arayalım
target_zip_name = 'vehicle-detection-clean.zip'
search_results = glob.glob(f'/content/drive/MyDrive/**/{target_zip_name}', recursive=True)

if search_results:
    gdrive_zip = search_results[0]
    print(f'✅ Veri seti bulundu: {gdrive_zip}')
    print('Yerel dizine kopyalanıyor ve arşiv açılıyor (Bu işlem dosya boyutundan dolayı birkaç dakika sürebilir)...')
    !cp "{gdrive_zip}" /content/ua_detrac_yolo.zip
    !unzip -q /content/ua_detrac_yolo.zip -d /content/ua_detrac_dataset
    print('✅ Veri seti başarıyla /content/ua_detrac_dataset dizinine açıldı!')
else:
    print(f'❌ HATA: Google Drive içerisinde "{target_zip_name}" dosyası bulunamadı.')
    print('Lütfen dosya adını kontrol edin ve Drive\'ınızda doğru konumda olduğundan emin olun.')

✅ Veri seti bulundu: /content/drive/MyDrive/Colab Notebooks/vehicle_detection/vehicle-detection-clean.zip
Yerel dizine kopyalanıyor ve arşiv açılıyor (Bu işlem dosya boyutundan dolayı birkaç dakika sürebilir)...
✅ Veri seti başarıyla /content/ua_detrac_dataset dizinine açıldı!


## 5. Veri Kümesi İçeriği ve Araç Türlerinin Analizi 📊
Drive'dan başarıyla açılan klasör yapısını inceleyerek hangi sınıfların, klasörlerin ve görsel tiplerinin bulunduğunu detaylandırıyoruz.

In [13]:
import os
import glob
from collections import Counter
import yaml

# Veri seti ana dizinini tespit et
possible_roots = [
    '/content/ua_detrac_dataset/vehicle-detection-clean',
    '/content/ua_detrac_dataset'
]
dataset_root = None
for pr in possible_roots:
    if os.path.exists(os.path.join(pr, 'train')):
        dataset_root = pr
        break

if not dataset_root:
    found = glob.glob('/content/ua_detrac_dataset/**/train', recursive=True)
    if found:
        dataset_root = os.path.dirname(found[0])

print("=" * 68)
print("     📊 YENİ TRAFİK VERİ SETİ DETAYLI ANALİZ RAPORU")
print("=" * 68)

if not dataset_root or not os.path.exists(dataset_root):
    print("❌ Veri seti dizini bulunamadı! Lütfen Adım 4'teki indirme/açma hücresini çalıştırın.")
else:
    print(f"📁 Veri Seti Kök Dizini: {dataset_root}\n")
    
    # Varsayılan sınıf sözlüğü
    class_names_map = {0: 'car', 1: 'motorcycle', 2: 'bus', 3: 'truck'}
    
    # Varsa data.yaml dosyasından sınıf isimlerini kontrol et
    yaml_candidates = glob.glob(f"{dataset_root}/*.yaml") + glob.glob("/content/ua_detrac_dataset/*.yaml")
    if yaml_candidates:
        try:
            with open(yaml_candidates[0], 'r', encoding='utf-8') as yf:
                y_data = yaml.safe_load(yf)
                if 'names' in y_data:
                    if isinstance(y_data['names'], list):
                        class_names_map = {i: n for i, n in enumerate(y_data['names'])}
                    elif isinstance(y_data['names'], dict):
                        class_names_map = {int(k): v for k, v in y_data['names'].items()}
                    print(f"📄 Bulunan data.yaml'dan okunan sınıflar: {class_names_map}\n")
        except:
            pass

    total_overall_boxes = 0
    overall_class_counts = Counter()
    split_summaries = {}

    for split in ['train', 'val']:
        split_dir = os.path.join(dataset_root, split)
        if not os.path.exists(split_dir):
            continue
            
        img_dir = os.path.join(split_dir, 'images')
        lbl_dir = os.path.join(split_dir, 'labels')
        
        img_files = glob.glob(f"{img_dir}/*.jpg") + glob.glob(f"{img_dir}/*.png") + glob.glob(f"{img_dir}/*.jpeg")
        lbl_files = glob.glob(f"{lbl_dir}/*.txt")
        
        class_counts = Counter()
        total_boxes = 0
        
        for lf in lbl_files:
            try:
                with open(lf, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            cls_id = int(float(parts[0]))
                            class_counts[cls_id] += 1
                            total_boxes += 1
            except:
                pass
                
        overall_class_counts.update(class_counts)
        total_overall_boxes += total_boxes
        split_summaries[split] = (len(img_files), len(lbl_files), total_boxes, class_counts)
        
        print(f"📌 [{split.upper()} KÜMESİ]")
        print(f"   • Toplam Görsel Sayısı: {len(img_files):,}")
        print(f"   • Toplam Etiket Dosyası: {len(lbl_files):,}")
        print(f"   • Toplam Bounding Box (Kutu): {total_boxes:,}")
        print(f"   • Sınıf Bazlı Araç Sayıları:")
        for cid in sorted(class_counts.keys()):
            cname = class_names_map.get(cid, f"class_{cid}")
            pct = (class_counts[cid] / total_boxes * 100) if total_boxes > 0 else 0.0
            print(f"      - ID {cid} ({cname:<12}): {class_counts[cid]:>7,} adet  (%{pct:>5.1f})")
        print("-" * 68)

    print("\n🎯 [GENEL TOPLAM]")
    total_imgs = sum(s[0] for s in split_summaries.values())
    print(f"   • Toplam Görsel: {total_imgs:,}")
    print(f"   • Toplam Etiketlenen Araç: {total_overall_boxes:,}")
    print(f"   • Sınıf Dağılımı:")
    for cid in sorted(overall_class_counts.keys()):
        cname = class_names_map.get(cid, f"class_{cid}")
        pct = (overall_class_counts[cid] / total_overall_boxes * 100) if total_overall_boxes > 0 else 0.0
        print(f"      - Sınıf {cid} ({cname:<12}): {overall_class_counts[cid]:>7,} adet  (%{pct:>5.1f})")
    print("=" * 68)


/content/ua_detrac_yolo.zip bulundu, arşiv açılıyor...
replace /content/ua_detrac_dataset/vehicle-detection-clean/.venv/Lib/site-packages/anyio/abc/_eventloop.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: ✅ Veri seti başarıyla açıldı!


---


## 5. data.yaml Konfigürasyonunu Doğrulama
Veri setinin dizinlerini ve 4 araç sınıfını (`car`, `motorcycle`, `bus`, `truck`) hazırlıyoruz.


In [ ]:
import os

# Gerçek verinin çıkartıldığı klasörü dinamik tespit ediyoruz
possible_roots = [
    '/content/ua_detrac_dataset/vehicle-detection-clean',
    '/content/ua_detrac_dataset'
]
dataset_base_path = '/content/ua_detrac_dataset/vehicle-detection-clean'
for pr in possible_roots:
    if os.path.exists(os.path.join(pr, 'train')):
        dataset_base_path = pr
        break

yaml_content = f"""# Vehicle Detection YOLO Dataset Configuration
path: {dataset_base_path}
train: train/images
val: val/images
test: val/images

nc: 4
names:
  0: car
  1: motorcycle
  2: bus
  3: truck
"""

yaml_path = '/content/ua_detrac_dataset/data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip() + '\n')

train_img_dir = os.path.join(dataset_base_path, 'train/images')
val_img_dir = os.path.join(dataset_base_path, 'val/images')

train_count = len(os.listdir(train_img_dir)) if os.path.exists(train_img_dir) else 0
val_count = len(os.listdir(val_img_dir)) if os.path.exists(val_img_dir) else 0

print('✅ Veri Seti Konfigürasyonu Güncellendi ve Doğrulandı:')
print(f' - dataset base path:         {dataset_base_path}')
print(f' - Train (Eğitim) Görselleri: {train_count}')
print(f' - Val (Doğrulama) Görselleri: {val_count}')
print(f' - Hazırlanan data.yaml Yolu:  {yaml_path}')


## 6. YOLO11n Modelinin Başlatılması ve Eğitimi (Training) 🚀
45.000+ gerçek araç verisiyle **YOLO11n (Nano - 3.1M Parametre)** modelini eğitiyoruz.

* **Model:** `yolo11n.pt` (Nano)
* **Epochs:** 30
* **imgsz:** 640
* **batch:** 16 (GPU için optimize)

In [ ]:
import os
import torch
from ultralytics import YOLO

# Donanım kontrolü
device_arg = 0 if torch.cuda.is_available() else 'cpu'
print(f'Kullanılan donanım: {device_arg} (CUDA: {torch.cuda.is_available()})')

# Önceden eğitilmiş YOLO11n (Nano) modelini yüklüyoruz (İsteğiniz doğrultusunda)
model = YOLO('yolo11n.pt')
print('YOLO11n nano modeli yüklendi. UA-DETRAC üzerinde eğitim başlatılıyor...')

results = model.train(
    data='/content/ua_detrac_dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=device_arg,
    project='runs/train',
    name='yolo11n_uad_exp',
    patience=10,
    save=True,
    plots=True,
    seed=42,
    verbose=True
)

## 7. Modelin Test Kümesinde Değerlendirilmesi (Evaluation)
Eğitilen en iyi ağırlık (`best.pt`) ile 1.594 adetlik bağımsız test kümesi üzerinde başarıyı ölçüyoruz.


In [ ]:
import os
from ultralytics import YOLO

# Test kümesi değerlendirmesi
metrics = model.val(split='test')

print('\n' + '=' * 50)
print('   UA-DETRAC TEST KÜMESİ BAŞARI SONUÇLARI')
print('=' * 50)
print(f"mAP@50:     {metrics.box.map50:.4f} (%{metrics.box.map50 * 100:.2f})")
print(f"mAP@50-95:  {metrics.box.map:.4f} (%{metrics.box.map * 100:.2f})")
print(f"Precision:  {metrics.box.mp:.4f} (%{metrics.box.mp * 100:.2f})")
print(f"Recall:     {metrics.box.mr:.4f} (%{metrics.box.mr * 100:.2f})")
print('=' * 50)


## 8. Eğitim Grafikleri ve Karışıklık Matrisi
Kayıp grafikleri (`results.png`), Precision-Recall eğrisi ve Confusion Matrix sonuçlarını görüntülüyoruz.


In [ ]:
import os
import glob
from IPython.display import Image, display

# Yol ismi yolo11n_uad_exp olarak güncellendi
plots = [
    'runs/train/yolo11n_uad_exp/results.png',
    'runs/train/yolo11n_uad_exp/confusion_matrix_normalized.png',
    'runs/train/yolo11n_uad_exp/BoxPR_curve.png'
]

for p in plots:
    if os.path.exists(p):
        print(f'Görsel: {os.path.basename(p)}')
        display(Image(filename=p, width=700))
    else:
        print(f'Grafik bulunamadı (Eğitim tamamlanmamış olabilir): {p}')

## 9. Gerçek Trafik Test Fotoğrafı Üzerinde Test (Inference)
Özel trafik fotoğrafımız (`test_samples/user_test_photo.jpg`) üzerinde çıkarım yapıyoruz.


In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

custom_photo = 'test_samples/user_test_photo.jpg'

if os.path.exists(custom_photo):
    print('Kıyaslama testi başlatılıyor...')

    # 1. Base (Eğitilmemiş Ham) YOLO11n modelini yükle ve tahmin et
    base_model = YOLO('yolo11n.pt')
    base_results = base_model.predict(source=custom_photo, conf=0.25, save=True, project='runs/detect', name='base_model_result', exist_ok=True)

    # 2. Bizim Eğittiğimiz Özel YOLO11n modelini yükle ve tahmin et
    trained_model_path = 'runs/train/yolo11n_uad_exp/weights/best.pt'
    if os.path.exists(trained_model_path):
        trained_model = YOLO(trained_model_path)
        trained_results = trained_model.predict(source=custom_photo, conf=0.25, save=True, project='runs/detect', name='trained_model_result', exist_ok=True)

        # Görselleri bul
        base_imgs = glob.glob('runs/detect/base_model_result/*.*')
        trained_imgs = glob.glob('runs/detect/trained_model_result/*.*')

        if base_imgs and trained_imgs:
            fig, axes = plt.subplots(1, 2, figsize=(20, 10))
            axes[0].imshow(Image.open(base_imgs[0]))
            axes[0].set_title('Orijinal (Base Pretrained) YOLO11n Çıktısı', fontsize=16)
            axes[0].axis('off')

            axes[1].imshow(Image.open(trained_imgs[0]))
            axes[1].set_title('Yeni Veriyle Eğitilmiş Özel YOLO11n Çıktısı', fontsize=16)
            axes[1].axis('off')

            plt.tight_layout()
            
            # Karşılaştırma görselini zip içine girmesi için eğitim klasörüne kaydet
            out_comp = 'runs/train/yolo11n_uad_exp/custom_test_comparison.jpg'
            plt.savefig(out_comp, bbox_inches='tight', dpi=150)
            print(f'✅ Karşılaştırma görseli kaydedildi: {out_comp}')
            plt.show()
    else:
        print(f'❌ Eğitilmiş model ağırlığı bulunamadı! Lütfen önce eğitim hücresini (Adım 6) çalıştırın. Aranan yol: {trained_model_path}')
else:
    print(f'❌ Test görseli bulunamadı: {custom_photo}')


## 10. Sonuçların Otomatik Paketlenmesi ve İndirilmesi 📦
Eğitilen `best.pt` ağırlık dosyasını ve tüm grafikleri tek bir `yolo11_results.zip` dosyası haline getirip indirir.


In [ ]:
import os
import shutil
from google.colab import files

# Yol ismi yolo11n_uad_exp olarak güncellendi
exp_dir = 'runs/train/yolo11n_uad_exp'
best_weights = f'{exp_dir}/weights/best.pt'
if os.path.exists(best_weights):
    size_mb = os.path.getsize(best_weights) / (1024 * 1024)
    print(f'✅ Yeni model ağırlığı hazır: {best_weights} ({size_mb:.2f} MB)')

print('📦 Sonuçlar paketleniyor ve yolo11_results.zip indiriliyor...')
shutil.make_archive('yolo11_results', 'zip', exp_dir)
files.download('yolo11_results.zip')
print('🚀 yolo11_results.zip indirme başlatıldı!')